# Alexiou Stamatios - 2020030158 - Assignment 1

# Exercise 4

In order to describe the game/puzzle Unruly as a search problem and come up with a solution for a specific instance, we can follow the steps below:

---

<h2>Defining the Search Space:</h2>

1. States: Every state consists of the nxm board with some positions colored black and some white. A state is complete if every position is colored either black or white.

2. Initial state: The initial board as given in the problem, where some positions are already colored black and/or white while the rest are colorless.

3. Goal State: A situation in which all the squares are colored according to the rules of the game:

  i) anytime there is no three consecutive squares of the same color held horizontally or vertically.

  ii) each row and column has the same number of black and white squares.

---

<h2>Systematic Search method:</h2>

The implemented code solves the "Unruly" puzzle using the Backtracking search algorithm. The idea of backtracking is a systematic search through all possible solutions, step by step, and when it comes to a conflict choice that contradicts the puzzle rules-it "backtracks" and tries another option.

How Backtracking Works:

<li>Recursion: The algorithm goes through each cell in the board, trying to fill it with one of two possible values ('b' for black or 'w' for white). </li>

<li>Validity Check: Once a value is placed in a cell, the algorithm will check whether the placement is valid which means the board doesn't contain three cells in a row of the same color within a row or column.</li>

<li>Backtracking: If it is an invalid placement or if the next step doesn't yield a solution, then the algorithm "backtracks" and changes the last placed number with another one from its possible values.</li>

<li>Goal: The aim is to fill up the board based on the rules of the puzzle and obtain a solution, if available. In the case where no solution is present, it stops searching.</li>

Additional Details:

<li>Maximum Node Limit: The algorithm is limited to searching a specified maximum amount of nodes that can be expanded; if this limit is surpassed, the search will stop with no solution found.</li>

<li>Goal State: This solution is valid when there is an equal number of 'b' and 'w' cells in every row and column. </li>

---

<h2>Heuristic Functions (Bonus Part):</h2>

The algorithm uses A* search, a graph traversal strategy that prioritizes nodes based on a cost function
𝑓
(
𝑠
)
=
𝑔
(
𝑠
)
+
ℎ
(
𝑠
)
, where
𝑔
(
𝑠
)
is the cost to reach the node, and
ℎ
(
𝑠
)
is the heuristic estimate of reaching the goal.

The heuristic function calculates:

<li>Rule Violations: Counts occurrences of three consecutive same-colored cells horizontally or vertically.</li>

<li>Color Imbalances: Measures the difference in the count of black and white cells across rows and columns.</li>

This combination ensures the solver effectively prioritizes states closer to the goal while minimizing constraint violations, leading to faster convergence to a valid solution.

---


**Run this script to generate a "random instance" of a board to solve.**

In [ ]:
"""
The approach to creating "random" puzzle instances that guarantee a solution is designed to ensure both balance and variety while maintaining solvability.
It starts by laying out a balanced board, where 'b' (black) and 'w' (white) are carefully placed in alternating quadrants, ensuring an equal distribution of colors across the board.
To add some randomness without disrupting this balance, rows and columns are shuffled within their respective halves.
We do not forget to check that no three consecutive cells of the same color appear in any row or column.
To make the puzzle more engaging, some of the cells are left uncolored, creating a partially completed board that the solver must fill in.
Finally, the board is encoded into a compact format.

This way the instances created are not 100% random but do preserve some randomness that creates different instances of a solvable board n*m.
"""
import random

def generate_balanced_board(n, m):
    # Ensure the board dimensions are even and >= 6 (for balance)
    assert n % 2 == 0 and m % 2 == 0 and n >= 6 and m >= 6

    # Initialize an empty board with the specified dimensions (n x m)
    board = [['' for _ in range(m)] for _ in range(n)]
    half_n = n // 2  # Half the number of rows
    half_m = m // 2  # Half the number of columns

    # Fill the board in a balanced way with 'b' and 'w' (black and white)
    for i in range(half_n):
        for j in range(half_m):
            # Place 'b' in the top-left and bottom-right quadrants, 'w' in the top-right and bottom-left
            board[i][j] = 'b'
            board[i + half_n][j] = 'w'
            board[i][j + half_m] = 'w'
            board[i + half_n][j + half_m] = 'b'

    # Shuffle the rows within the top and bottom halves of the board
    for i in range(half_n):
        swap_row = random.randint(0, half_n - 1)  # Randomly select a row to swap
        board[i], board[swap_row] = board[swap_row], board[i]  # Swap rows
        # Swap corresponding rows in the bottom half of the board
        board[i + half_n], board[swap_row + half_n] = board[swap_row + half_n], board[i + half_n]

    # Shuffle the columns within the left and right halves of the board
    for j in range(half_m):
        swap_col = random.randint(0, half_m - 1)  # Randomly select a column to swap
        for i in range(n):
            board[i][j], board[i][swap_col] = board[i][swap_col], board[i][j]  # Swap columns
            # Swap corresponding columns in the right half of the board
            board[i][j + half_m], board[i][swap_col + half_m] = board[i][swap_col + half_m], board[i][j + half_m]

    return board

def enforce_no_three_consecutive(board, n, m):
    # Ensure no three consecutive cells have the same color in rows
    for i in range(n):
        for j in range(m - 2):
            if board[i][j] == board[i][j + 1] == board[i][j + 2]:
                swap_col = (j + 3) % m  # Swap with a column far enough to avoid three consecutive same colors
                if board[i][swap_col] != board[i][j]:
                    board[i][j + 2], board[i][swap_col] = board[i][swap_col], board[i][j + 2]
                else:
                    board[i][j + 2] = 'b' if board[i][j] == 'w' else 'w'  # Change to the opposite color

    # Ensure no three consecutive cells have the same color in columns
    for j in range(m):
        for i in range(n - 2):
            if board[i][j] == board[i + 1][j] == board[i + 2][j]:
                swap_row = (i + 3) % n  # Swap with a row far enough to avoid three consecutive same colors
                if board[swap_row][j] != board[i][j]:
                    board[i + 2][j], board[swap_row][j] = board[swap_row][j], board[i + 2][j]
                else:
                    board[i + 2][j] = 'b' if board[i][j] == 'w' else 'w'  # Change to the opposite color

def uncolor_board(board, uncolor_fraction):
    n = len(board)
    m = len(board[0])
    total_cells = n * m
    num_uncolored = int(total_cells * uncolor_fraction)  # Number of cells that should be uncolored

    # Create a list of all cell positions
    cells = [(i, j) for i in range(n) for j in range(m)]
    random.shuffle(cells)  # Shuffle the cell positions randomly

    # Set the chosen cells to be uncolored ('.')
    for i, j in cells[:num_uncolored]:
        board[i][j] = '.'

    return board

def encode_board(board):
    # Flatten the board into a single list of cells
    n = len(board)
    m = len(board[0])
    flat_board = [color for row in board for color in row]
    encoded = ""
    last_pos = -1  # Track the position of the last colored cell

    # Encode the board as a sequence of characters
    for i, color in enumerate(flat_board):
        if color != '.':
            distance = i - last_pos  # Calculate the distance from the last colored cell
            if color == 'b':
                encoded += chr(ord('a') + distance - 1).upper()  # Encode 'b' using an uppercase letter
            elif color == 'w':
                encoded += chr(ord('a') + distance - 1)  # Encode 'w' using a lowercase letter
            last_pos = i

    # Add the remaining distance at the end of the encoding
    encoded += chr(ord('a') + len(flat_board) - last_pos - 1)
    return f"{n}x{m}:{encoded}"

def save_puzzle_instance(filename, instance):
    # Save the encoded puzzle instance to a file
    with open(filename, 'w') as f:
        f.write(instance)

def create_puzzle_instance(n, m, uncolor_fraction):
    # Generate a balanced board
    board = generate_balanced_board(n, m)
    # Ensure no three consecutive same colors in rows and columns
    enforce_no_three_consecutive(board, n, m)
    # Uncolor a fraction of the board
    initial_board = uncolor_board(board, uncolor_fraction)
    # Encode the initial board into a string format
    encoded = encode_board(initial_board)
    return initial_board, encoded

def print_board(board):
    for row in board:
        print(" ".join(row))

def main():
    n = int(input("Enter the number of rows (n): "))
    m = int(input("Enter the number of columns (m): "))
    uncolor_fraction = float(input("Enter the uncoloring fraction (0.0-1.0): "))
    output_filename = input("Enter the output filename: ")

    if n % 2 != 0 or m % 2 != 0 or n < 6 or m < 6:
        print("Both n and m must be even and greater than or equal to 6.")
        return

    initial_board, instance = create_puzzle_instance(n, m, uncolor_fraction)

    print("---------------------------------------------------")
    print("Initial Board:")
    print("---------------------------------------------------")
    print_board(initial_board)
    print("---------------------------------------------------")
    print(f"\nEncoded Instance: {instance}")

    save_puzzle_instance(output_filename, instance)
    print(f"\nPuzzle instance saved to {output_filename}")

if __name__ == "__main__":
    main()

Enter the number of rows (n): 6
Enter the number of columns (m): 6
Enter the uncoloring fraction (0.0-1.0): 0
Enter the output filename: puzzleee.txt
---------------------------------------------------
Initial Board:
---------------------------------------------------
b b w b w w
b b w b w w
w w b w b b
b b w b w w
w w b w b b
w w b w b b
---------------------------------------------------

Encoded Instance: 6x6:AAaAaaAAaAaaaaAaAAAAaAaaaaAaAAaaAaAAa

Puzzle instance saved to puzzleee.txt


**Function to decode the input file according to the specified format.**

In [ ]:
def decode_input(file_name):
    with open(file_name, 'r') as file:
        # Read the first line from the input file and remove any whitespace
        contents = file.readline().strip()

    # Get from contents the dimensions and the init_state by splitting at ":"
    dimensions, initial_state = contents.split(':')

    # Get the number of rows (n) and columns (m) by splitting at "x"
    dimensions_parts = dimensions.split('x')
    n = int(dimensions_parts[0])
    m = int(dimensions_parts[1])

    # Initialize an empty board with dots (uncolored cells)
    board = []
    for i in range(n):
        row = ['.' for _ in range(m)]
        board.append(row)

    curr_pointer = -1
    for index, char in enumerate(initial_state):
        # Calculate the distance from the previous position based on the character
        # ord(letter) starts at 96 for 'a' and so on. More info: https://stackoverflow.com/questions/4528982/convert-alphabet-letters-to-number-in-python
        distance = ord(char.lower()) - ord('a') + 1
        curr_pointer += distance

        # Check if the current pointer is outside the board bounds
        if curr_pointer == n * m:
            if index == len(initial_state) - 1:
                # If this is the last character and it goes out at [n][m+1] it's correctly formatted
                print(f"\nLetter: {char} indicates the end of the board.\n")
            else:
                raise ValueError("Invalid encoding: character moves out of bounds before the end.")
            break

        # Calculate the row and column based on the current pointer
        curr_row = curr_pointer // m
        curr_col = curr_pointer % m

        # Using 'b' for black cells and 'w' for white cells
        if char.isupper():
            board[curr_row][curr_col] = 'b'  # black
        else:
            board[curr_row][curr_col] = 'w'  # white

    # Return the initialized board and its dimensions
    return board, n, m


**Function to encode the output according to the specified format.**

In [ ]:
def encode_output(board):
    n = len(board)
    m = len(board[0])
    solution = f"{n}x{m}:"

    last_position = -1
    for i in range(n):
        for j in range(m):
            # Calculate the current pointer row by row
            curr_pointer = i * m + j
            # Only process cells that are filled
            if board[i][j] != '.':
                # Calculate the distance between the current and last filled position
                distance = curr_pointer - last_position - 1
                # Encode 'b' (black) as uppercase and 'w' (white) as lowercase
                if board[i][j] == 'b':
                    solution += chr(ord('A') + distance)
                else:
                    solution += chr(ord('a') + distance)
                # Update the last position to the current one
                last_position = curr_pointer

    solution += chr(ord('a'))  # Last 'a' indicates EOF

    return solution

**Implementing the Systematic Search Algorithm (Backtracking)**

In [ ]:
import time

class UnrulyBacktrackingSolver:
    def __init__(self, board, n, m, max_nodes):
        """
        The Solver object gets initialized with its real board, dimensions, and maximum number of nodes that should be expanded.
          board: 2D list representing the initial state of the board.
          n: Number of rows in the board.
          m: Number of columns in the board.
          max_nodes: Maximum number of nodes to expand during the search.
        """
        self.board = board
        self.n = n
        self.m = m
        self.max_nodes = max_nodes
        self.node_count = 0

    def is_valid_option(self, board, row, col, color):
        """
        Check whether putting 'color' in (row, col) holds according to the rules of the Unruly puzzle.
        No three adjacent cells in a row or a column should be of the same color.
          board: Current the board state.
          row: Row index that needs checking.
          col: Column index that needs checking.
          color: The color to be placed ('b' for black, 'w' for white).
          return: True if valid, False otherwise.
        """
        # Check horizontally
        if col >= 2 and board[row][col-1] == color and board[row][col-2] == color:
            return False
        # Check vertically
        if row >= 2 and board[row-1][col] == color and board[row-2][col] == color:
            return False
        # Check horizontal between two same colors
        if col >= 1 and col < self.m - 1 and board[row][col-1] == color and board[row][col+1] == color:
            return False
        # Check vertical between two same colors
        if row >= 1 and row < self.n - 1 and board[row-1][col] == color and board[row+1][col] == color:
            return False
        return True

    def solve(self):
        """
        Solve the Unruly puzzle by backtracking.
          return: True if a solution is found, False otherwise.
        """
        start_time = time.time()  # Start the timer
        success = self.backtrack(self.board, 0, 0)
        end_time = time.time()  # End the timer

        if success:
            print("\nSolution found:")
            for row in self.board:
                print(''.join(row))
        else:
            print("\nNo solution found or node limit reached.")

        print(f"\nTime taken: {end_time - start_time:.2f} seconds")
        print(f"\nNodes expanded: {self.node_count}")

        return success

    def backtrack(self, board, row, col):
        """
        Recursive backtracking method to solve the board.
          board: Current state of the board.
          row: Current row index.
          col: Current column index.
          return: True if a valid solution is found, False otherwise.
        """
        if self.node_count >= self.max_nodes:
            return False

        self.node_count += 1

        if row == self.n:  # If we've reached the end of the board
            return self.is_goal(board)

        next_row, next_col = (row, col + 1) if col + 1 < self.m else (row + 1, 0)

        if board[row][col] != '.':  # Skip pre-filled cells
            return self.backtrack(board, next_row, next_col)

        for color in 'bw':
            if self.is_valid_option(board, row, col, color):
                board[row][col] = color
                if self.backtrack(board, next_row, next_col):
                    return True
                board[row][col] = '.'  # Reset cell

        return False

    def is_goal(self, board):
        """
        Check if the current board is a valid solution.
          board: Current state of the board.
          return: True if the board is a valid solution, False otherwise.
        """
        # Check for equal number of 'b' and 'w' in each row
        for row in board:
            if row.count('b') != row.count('w'):
                return False
        # Check for equal number of 'b' and 'w' in each column
        for col in zip(*board):
            if col.count('b') != col.count('w'):
                return False
        return True

    def save_solution(self, output_file):
        """
        Save the solved board to the output file.
        """
        solution = encode_output(self.board)
        with open(output_file, 'w') as file:
            file.write(solution)
        print(f"Solution saved to {output_file}")

def main(input_file, output_file, max_nodes):
    """
    Main function to read input, solve the puzzle, and save the output.
      input_file: Path to the input file containing the initial board state.
      output_file: Path to the output file where the solution will be saved.
      max_nodes: Maximum number of nodes to expand during the search.
    """
    board, n, m = decode_input(input_file)
    print("Initial Board:")
    for row in board:
        print(' '.join(row))

    solver = UnrulyBacktrackingSolver(board, n, m, max_nodes)
    if solver.solve():
        solver.save_solution(output_file)

if __name__ == "__main__":
    input_file = input("Enter the input file name: ")
    output_file = input("Enter the output file name: ")
    max_nodes = int(input("Enter the maximum number of nodes to expand: "))
    main(input_file, output_file, max_nodes)


Enter the input file name: puzzle1.txt
Enter the output file name: output1.txt
Enter the maximum number of nodes to expand: 4693486934869385

Letter: c indicates the end of the board.

Initial Board:
. . w . . w w . . .
. b . b . . w b . .
. . b . . . . . b .
. . w . . w w b w w
b . w . . . w . . .
. . b w . . . w . b
w . . . w b . w . b
b . w . . . w . . w
. . b . . . . . b .
. . . . . . b w . .

Solution found:
bbwbbwwbww
bbwbbwwbww
wwbwwbbwbb
bbwbbwwbww
bbwbbwwbww
wwbwwbbwbb
wwbwwbbwbb
bbwbbwwbww
wwbwwbbwbb
wwbwwbbwbb

Time taken: 814.88 seconds

Nodes expanded: 510864077
Solution saved to output1.txt


**Bonus:**

**Implementing the Heuristic Search Algorithm (A* algorithm)**

In [ ]:
import time
import heapq

class UnrulyHeuristicSolver:
    def __init__(self, board, n, m, max_nodes):
        """
        Solver with heuristics for Unruly puzzle.
          board: 2D list representing the initial state of the board.
          n: Number of rows in the board.
          m: Number of columns in the board.
          max_nodes: Maximum number of nodes to expand during the search.
        """
        self.board = board
        self.n = n
        self.m = m
        self.max_nodes = max_nodes
        self.node_count = 0

    def is_valid_option(self, board, row, col, color):
        """
        Check whether putting 'color' in (row, col) is according to the rules of the Unruly puzzle.
        """
        # Check horizontal triplets to the left
        if col >= 2 and board[row][col-1] == color and board[row][col-2] == color:
            return False
        # Check vertical triplets above
        if row >= 2 and board[row-1][col] == color and board[row-2][col] == color:
            return False
        # Check horizontal triplets around the cell
        if col >= 1 and col < self.m - 1 and board[row][col-1] == color and board[row][col+1] == color:
            return False
        # Check vertical triplets around the cell
        if row >= 1 and row < self.n - 1 and board[row-1][col] == color and board[row+1][col] == color:
            return False
        return True

    def calculate_heuristic(self, board):
        """
        Calculate the heuristic value for the current board state.
        """
        violations = 0
        # Count violations of triplets in rows
        for row in board:
            row_str = ''.join(row)
            violations += row_str.count('bbb') + row_str.count('www')

        # Count violations of triplets in columns
        for col in zip(*board):
            col_str = ''.join(col)
            violations += col_str.count('bbb') + col_str.count('www')

        imbalance = 0
        # Calculate imbalance for rows (difference in counts of 'b' and 'w')
        for row in board:
            imbalance += abs(row.count('b') - row.count('w'))

        # Calculate imbalance for columns
        for col in zip(*board):
            imbalance += abs(col.count('b') - col.count('w'))

        return violations + imbalance

    def solve(self):
        """
        Solve the Unruly puzzle using A* search with heuristics.
        """
        start_time = time.time()
        success = self.a_star_search()
        end_time = time.time()

        if success:
            print("\nSolution found:")
            for row in self.board:
                print(''.join(row))
        else:
            print("\nNo solution found or node limit reached.")

        print(f"\nTime taken: {end_time - start_time:.2f} seconds")
        print(f"\nNodes expanded: {self.node_count}")

        return success

    def a_star_search(self):
        """
        A* search algorithm with heuristics.
        """
        # Priority queue to store board states with their heuristic values
        priority_queue = []
        heapq.heappush(priority_queue, (0, self.board, 0, 0))

        while priority_queue:
            if self.node_count >= self.max_nodes:
                return False

            # Get the board state with the lowest heuristic value
            priority, board, row, col = heapq.heappop(priority_queue)
            self.node_count += 1

            if row == self.n:
                if self.is_goal(board):
                    self.board = board
                    return True
                continue

            # Determine the next cell to process
            next_row, next_col = (row, col + 1) if col + 1 < self.m else (row + 1, 0)

            if board[row][col] != '.':
                heapq.heappush(priority_queue, (priority, board, next_row, next_col))
                continue

            # Try placing 'b' or 'w' in the current cell
            for color in 'bw':
                if self.is_valid_option(board, row, col, color):
                    # Create a new board with the current move
                    new_board = [list(row) for row in board]
                    new_board[row][col] = color
                    # Calculate the heuristic value for the new board
                    h_value = self.calculate_heuristic(new_board)
                    heapq.heappush(priority_queue, (priority + h_value, new_board, next_row, next_col))

        return False

    def is_goal(self, board):
        """
        Check if the current board is a valid solution.
        """
        for row in board:
            if row.count('b') != row.count('w'):
                return False

        for col in zip(*board):
            if col.count('b') != col.count('w'):
                return False

        return True

    def save_solution(self, output_file):
        """
        Save the solved board to the output file.
        """
        solution = encode_output(self.board)
        with open(output_file, 'w') as file:
            file.write(solution)
        print(f"Solution saved to {output_file}")

def main(input_file, output_file, max_nodes):
    board, n, m = decode_input(input_file)
    print("Initial Board:")

    for row in board:
        print(' '.join(row))

    solver = UnrulyHeuristicSolver(board, n, m, max_nodes)

    if solver.solve():
        solver.save_solution(output_file)

if __name__ == "__main__":
    input_file = input("Enter the input file name: ")
    output_file = input("Enter the output file name: ")
    max_nodes = int(input("Enter the maximum number of nodes to expand: "))
    main(input_file, output_file, max_nodes)

Enter the input file name: puzzle1.txt
Enter the output file name: outputHeuristic.txt
Enter the maximum number of nodes to expand: 848484848484842

Letter: c indicates the end of the board.

Initial Board:
. . w . . w w . . .
. b . b . . w b . .
. . b . . . . . b .
. . w . . w w b w w
b . w . . . w . . .
. . b w . . . w . b
w . . . w b . w . b
b . w . . . w . . w
. . b . . . . . b .
. . . . . . b w . .

Solution found:
bwwbbwwbwb
wbwbwbwbbw
wwbwbwbwbb
bbwbbwwbww
bbwbwbwbww
wwbwbwbwbb
wbbwwbbwwb
bbwbwbwbww
wwbwbwbwbb
bwbwwbbwbw

Time taken: 63.56 seconds

Nodes expanded: 1014656
Solution saved to outputHeuristic.txt


<h2>Observations between the two Algorithms Implementation:</h2>

  We tested the algorithms for different board sizes and initial states.
  During these testings we conducted the following results:

  1. Both backtracking and A* guarantee to find a solution if exists.

  2. Backtracking performed intrestingly well for board sizes 6x6, 8x8 with a uncolor_fraction greater than 0.5.

  3. For the test input given in the assignment, the backtracking algorithm took only 0.02s to solve the puzzle and expanded 7722 nodes. The A* took 0.14s and expanded 5569 nodes.

  4. In a 10x10 board instance generated with the first code cell:
              10x10:ccaEBcAEFdcaAaaAbdFadBadAbBAbdcCFHac
  The backtracking algorithm took 814.88s and expanded 5108640770 nodes while the A* took only 63.56s and expanded 1014656 nodes.

---

  Therefore, we can see that A* outperformed Backtracking Algorithm for bigger dimensions boards. Both algorithms gave a DIFFERENT (sometimes) solution with the A* giving the optimal one.           